In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import zipfile
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet, stopwords
from spellchecker import SpellChecker
from functools import lru_cache
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GridSearchCV
from gensim.models import Word2Vec
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold

# Preprocessing

In [2]:
with zipfile.ZipFile('../datasets/p00_tweets.zip', 'r') as z:
    for name in z.namelist():
        info = z.getinfo(name)

In [ ]:
def parse_file(filepath, class_name):
    """сплит текстов через запятые + заглавные буквы или .?! + заглавные буквы"""
    with open(filepath, encoding='utf-8') as f:
        text = f.read()
    
    parts = re.split(r',(?=\s*[A-Z])', text)
    
    result = []
    for p in parts:
        sub = re.split(r'(?<=[.!?])\s+(?=[A-Z])', p)
        result.extend(sub)
    
    result = [r.strip() for r in result if r.strip() and len(r.strip()) > 10]
    return pd.DataFrame({'text': result, 'type': class_name})

def strip_markers(text, markers):
    """удаляет маркеры из текста"""
    for m in markers:
        text = re.sub(rf'\s+{re.escape(m)}\s*$', '', text)

        text = re.sub(rf'[,.]\s*{re.escape(m)}\s*[,.]', '. ', text)
        text = re.sub(rf'\s+{re.escape(m)}\s+', ' ', text)
    return text.strip()

In [4]:
data_pos = parse_file('../datasets/p00_tweets/processedPositive.csv', 'positive')
data_neg = parse_file('../datasets/p00_tweets/processedNegative.csv', 'negative')
data_neu = parse_file('../datasets/p00_tweets/processedNeutral.csv', 'neutral')

markers = {
    'positive': ['happy', 'smile', 'crying with joy'],
    'negative': ['unhappy', 'sad', 'crying', 'silence', 'annoyance'],
    'neutral':  [],
}

data_pos['text'] = data_pos['text'].apply(lambda t: strip_markers(t, markers['positive']))
data_neg['text'] = data_neg['text'].apply(lambda t: strip_markers(t, markers['negative']))

data = pd.concat([data_pos, data_neg, data_neu], ignore_index=True)

print(data.shape)
print(data['type'].value_counts())

print("\n")
for t in data.sample(5, random_state=42)['text']:
    print(f"{t[:150]}")

(2927, 2)
type
neutral     1105
positive    1076
negative     746
Name: count, dtype: int64


How Modi won UP, dynasties demonetised, what the numbers say, and more.
Thanks for the recent follow Happy to connect have a great Thursday.,thanks :D
Calcutta High Court orders to probe sting operation.
I know how you feel, my views have gone done by about 100 monthly ,mane has been amazing ,senior snaps are starting to get so senti aw ,instant messag
I wish i could vote for you ,instant message so jealous okay but never mind haha bruno can wait finals first,when i'm enlisting can please turn up lik


In [ ]:
data

,text,type
0,"An inspiration in all aspects: Fashion, fitnes...",positive
1,Apka Apna Awam Ka Channel Frankline Tv Aam Adm...,positive
2,Beautiful album from the greatest unsung guit...,positive
3,Good luck to Rich riding for great project in ...,positive
4,Can you donate?,positive
...,...,...
2922,"IDFC official Vikram Limaye, former captain Di...",neutral
2923,Supreme Court names former CAG as head of 4-me...,neutral
2924,Court summons CM suspended BJP MP as accused i...,neutral
2925,Amulya Patnaik has been appointed new Delhi po...,neutral


после анализа текста найдены следующие проблемы:
- u заменяется на you даже просто внутри слов когда это не нужно
- n заменяется на and аналогично внутри слов
- рандомно появляются not applicable в тексте (как понял судя по всему скрытия каких то пользователей личнсотей или данных)
- %27 вместе ' например don%27t
- ну и проблема решенная ранее - маркеры текста в конце твитов
- также в тексте positive u заменяется на year old
- в некоторых моментах числа 2 заменены на to, и 4 на for

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['type'], test_size=0.2, random_state=42, stratify=data['type'])

# Experiments

In [6]:
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [7]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stops = set(stopwords.words('english'))
spell = SpellChecker()
for w in ['omg', 'lol', 'rt', 'via', 'amp', 'dont', 'cant', 'wont',
          'gonna', 'wanna', 'tbh', 'btw', 'afaik', 'idk', 'omfg',
          'ily', 'smh', 'tbt', 'ffs', 'af', 'fb', 'ig']:
    if len(w) > 2:
        spell.word_frequency.add(w)

In [11]:
def preprocess_tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

def preprocess_stem(text):
    words = preprocess_tokenizer(text)
    return [stemmer.stem(word) for word in words]

def preprocess_lem(text):
    words = preprocess_tokenizer(text)
    return [lemmatizer.lemmatize(w) for w in words]

def preprocess_stem_stop(text):
    words = preprocess_tokenizer(text)
    return [stemmer.stem(w) for w in words if w not in stops]

@lru_cache(maxsize=50000)
def _correct_word(word):
    """кеширует коррекции чтобы второй вызов был кратно быстрее"""
    if len(word) <= 2 or not word.isalpha():
        return word
    if word.lower() not in spell:
        corrected = spell.correction(word)
        return corrected if corrected else word
    return word

def preprocess_misspell(text):
    words = preprocess_tokenizer(text)
    return [_correct_word(w) for w in words]

def preprocess_lemma_misspell(text):
    fixed = preprocess_misspell(text)
    return [lemmatizer.lemmatize(w) for w in fixed]

In [12]:
def vectorization():
    return {
        'binary': CountVectorizer(binary=True, max_features=20000, ngram_range=(1, 2)),
        'counts': CountVectorizer(max_features=20000, ngram_range=(1, 2)),
        'tfidf': TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                                  sublinear_tf=True, stop_words='english'),
        'tfidf_no_stop': TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                                     sublinear_tf=True)
    }

In [13]:
def top_10(X_vec, texts, top_k=10, max_sim=0.99, min_words=4):
    texts = list(texts)
    seen = {}
    for i, t in enumerate(texts):
        if len(t.split()) < min_words:
            continue
        if t not in seen:
            seen[t] = i

    unique_idx = list(seen.values())
    unique_texts = [texts[i] for i in unique_idx]
    X_unique = X_vec[unique_idx]

    sim = cosine_similarity(X_unique)
    np.fill_diagonal(sim, 0)

    sim = np.where(sim < max_sim, sim, 0)
    
    i, j = np.triu_indices_from(sim, k=1)
    sims = sim[i, j]
    top = np.argsort(sims)[-top_k:][::-1]
    
    return pd.DataFrame({
        'rank':       range(1, top_k + 1),
        'similarity': sims[top].round(4),
        'tweet_1':    [unique_texts[i[k]] for k in top],
        'tweet_2':    [unique_texts[j[k]] for k in top],
    })

## Logistic Regression

In [105]:
preprocessors = {
    'tokenize': preprocess_tokenizer,
    'stemming': preprocess_stem,
    'lemmatization': preprocess_lem,
    'stem+stops': preprocess_stem_stop,
    'misspellings': preprocess_misspell,
    'misspeell+lemm': preprocess_lemma_misspell
}

vectorizer = vectorization()

res = {}
all_similarity = []
for prep_name, prep_func in preprocessors.items():
    X_train_prep = X_train.apply(lambda t: ' '.join(prep_func(t)))
    X_test_prep = X_test.apply(lambda t: ' '.join(prep_func(t)))

    for vec_name, vec in vectorizer.items():
        vec_copy = vec.__class__(**vec.get_params())

        X_train_vec = vec_copy.fit_transform(X_train_prep)
        X_test_vec = vec_copy.transform(X_test_prep)

        X_train_vec = X_train_vec.astype(np.float32)
        X_test_vec  = X_test_vec.astype(np.float32)

        model = LogisticRegression(max_iter=1000, class_weight='balanced')

        model.fit(X_train_vec, y_train)
        pred = model.predict(X_test_vec)
        acc = accuracy_score(y_test, pred)

        res[(prep_name, vec_name)] = acc

        df_sim = top_10(X_test_vec, X_test_prep)
        df_sim.insert(0, 'prep', prep_name)
        df_sim.insert(1, 'vec',  vec_name)
        all_similarity.append(df_sim)

        print(f"{prep_name}, {vec_name} done")

similarity_df = pd.concat(all_similarity, ignore_index=True)

tokenize, binary done
tokenize, counts done
tokenize, tfidf done
tokenize, tfidf_no_stop done
stemming, binary done
stemming, counts done
stemming, tfidf done
stemming, tfidf_no_stop done
lemmatization, binary done
lemmatization, counts done
lemmatization, tfidf done
lemmatization, tfidf_no_stop done
stem+stops, binary done
stem+stops, counts done
stem+stops, tfidf done
stem+stops, tfidf_no_stop done
misspellings, binary done
misspellings, counts done
misspellings, tfidf done
misspellings, tfidf_no_stop done
misspeell+lemm, binary done
misspeell+lemm, counts done
misspeell+lemm, tfidf done
misspeell+lemm, tfidf_no_stop done


In [106]:
similarity_df.to_csv('similarity_full.csv', index=False)
similarity_df

,prep,vec,rank,similarity,tweet_1,tweet_2
0,tokenize,binary,1,0.9271,thanks for the recent follow happy to connect ...,thanks for the recent follow happy to connect ...
1,tokenize,binary,2,0.9058,thanks for the recent follow happy to connect ...,thanks for the recent follow happy to connect ...
2,tokenize,binary,3,0.8840,thanks for the recent follow happy to connect ...,thanks for the recent follow happy to connect ...
3,tokenize,binary,4,0.8469,why are you whimper at me this is damn theres ...,ive just wrote tweets about but still didnt no...
4,tokenize,binary,5,0.8385,i want to see them holding the trophy anyways ...,i want to see them holding the trophy anyways ...
...,...,...,...,...,...,...
235,misspeell+lemm,tfidf_no_stop,6,0.8114,bring back the ginger curl talking to my over ...,and im still not sleeping talking to my over d...
236,misspeell+lemm,tfidf_no_stop,7,0.7998,why are you whimper at me this is damn there p...,i really enjoyed horizon zero dawn but no idea...
237,misspeell+lemm,tfidf_no_stop,8,0.7961,thanks for the recent follow happy to connect ...,thanks for the recent follow happy to connect ...
238,misspeell+lemm,tfidf_no_stop,9,0.7908,all i had wa a sweep hand puppet and the basta...,i really enjoyed horizon zero dawn but no idea...


In [34]:
df_res = pd.DataFrame([(k[0], k[1], v) for k, v in res.items()], columns=['preprocessing', 'vectorization', 'accuracy'])

pivot = df_res.pivot(index='preprocessing', columns='vectorization', values='accuracy')
print(pivot.round(4))

vectorization   binary  counts   tfidf  tfidf_no_stop
preprocessing                                        
lemmatization   0.7850  0.7884  0.7372         0.7952
misspeell+lemm  0.7918  0.7867  0.7389         0.7969
misspellings    0.7935  0.7901  0.7457         0.8055
stem+stops      0.7509  0.7457  0.7406         0.7526
stemming        0.7884  0.7884  0.7509         0.7918
tokenize        0.7850  0.7799  0.7321         0.8003


## Linear SVC

In [79]:
preprocessors = {
    'tokenize': preprocess_tokenizer,
    'stemming': preprocess_stem,
    'lemmatization': preprocess_lem,
    'stem+stops': preprocess_stem_stop,
}

vectorizer = vectorization()

res = {}
for prep_name, prep_func in preprocessors.items():
    X_train_prep = X_train.apply(lambda t: ' '.join(prep_func(t)))
    X_test_prep = X_test.apply(lambda t: ' '.join(prep_func(t)))

    for vec_name, vec in vectorizer.items():
        vec_copy = vec.__class__(**vec.get_params())

        X_train_vec = vec_copy.fit_transform(X_train_prep)
        X_test_vec = vec_copy.transform(X_test_prep)

        X_train_vec = X_train_vec.astype(np.float32)
        X_test_vec  = X_test_vec.astype(np.float32)

        model = LinearSVC(max_iter=1000, class_weight='balanced')

        model.fit(X_train_vec, y_train)
        pred = model.predict(X_test_vec)
        acc = accuracy_score(y_test, pred)

        res[(prep_name, vec_name)] = acc
        print(f"{prep_name}, {vec_name} done")

tokenize, binary done
tokenize, counts done
tokenize, tfidf done
tokenize, tfidf_no_stop done
stemming, binary done
stemming, counts done
stemming, tfidf done
stemming, tfidf_no_stop done
lemmatization, binary done
lemmatization, counts done
lemmatization, tfidf done
lemmatization, tfidf_no_stop done
stem+stops, binary done
stem+stops, counts done
stem+stops, tfidf done
stem+stops, tfidf_no_stop done


In [80]:
df_res = pd.DataFrame([(k[0], k[1], v) for k, v in res.items()], columns=['preprocessing', 'vectorization', 'accuracy'])

pivot = df_res.pivot(index='preprocessing', columns='vectorization', values='accuracy')
print(pivot.round(4))

vectorization  binary  counts   tfidf  tfidf_no_stop
preprocessing                                       
lemmatization  0.7765  0.7696  0.7389         0.8020
stem+stops     0.7116  0.7133  0.7594         0.7560
stemming       0.7833  0.7799  0.7543         0.7969
tokenize       0.7765  0.7765  0.7355         0.8020


# Let me try my best

после анализа текста найдены следующие проблемы:
- u заменяется на you даже просто внутри слов когда это не нужно
- n заменяется на and аналогично внутри слов
- рандомно появляются not applicable в тексте (как понял судя по всему скрытия каких то пользователей личнсотей или данных)
- %27 вместе ' например don%27t
- ну и проблема решенная ранее - маркеры текста в конце твитов
- также в тексте positive u заменяется на year old
- в некоторых моментах числа 2 заменены на to, и 4 на for

In [ ]:
def reverse_artifacts(word):
    if word != 'you' and not word.startswith('you') and 'you' in word:
        word = word.replace('you', 'u')
    
    if word != 'and' and not word.startswith('and') and 'and' in word:
        word = word.replace('and', 'n')
        
    return word

def preprocess_fix(text):
    text = text.replace('%27', "'").replace('&amp;', '&').replace('&gt;', '>')

    text = re.sub(r'\byear old\b', 'you', text)

    text = re.sub(r'\bnot applicable\w*', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    words = text.split()
    words = [reverse_artifacts(w) for w in words]
    return words

In [19]:
def preprocess_fix_misspell(text):
    fixed = preprocess_fix(text)
    return [_correct_word(w) for w in fixed]

def preprocess_fix_misspell_lem(text):
    fixed = preprocess_fix_misspell(text)
    return [lemmatizer.lemmatize(w) for w in fixed]

- char wb - символьный нграммы, внутри границ слова
- берем от трйоных до пятерных нграмм
- берем 30к наиболее значимых нграмм (полагаю бесполезно в нашей задаче)
- саблайниар - дефолт tf чтобы снижать веса частых слов

In [20]:
def build_hybrid(X_train_text, X_test_text):
    vec_w = TfidfVectorizer(
        analyzer='word', ngram_range=(1, 2),
        max_features=15000, sublinear_tf=True
    )
    X_train_w = vec_w.fit_transform(X_train_text).astype(np.float32)
    X_test_w  = vec_w.transform(X_test_text).astype(np.float32)

    vec_c = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(3, 5),
        max_features=25000, sublinear_tf=True
    )
    X_train_c = vec_c.fit_transform(X_train_text).astype(np.float32)
    X_test_c  = vec_c.transform(X_test_text).astype(np.float32)

    X_train = hstack([X_train_w, X_train_c])
    X_test  = hstack([X_test_w,  X_test_c])

    return X_train, X_test

In [69]:
def vectorization():
    return {
        'tfidf_no_stop': lambda: TfidfVectorizer(
            max_features=20000, ngram_range=(1, 2), sublinear_tf=True
        ),
        'char': lambda: TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3, 5),
            max_features=30000, sublinear_tf=True
        ),
        'hybrid': 'hybrid',
    }

In [ ]:
preprocessors = {
    'fix': preprocess_fix,
    'fix+misspell': preprocess_fix_misspell,
    'fix+misspell+lemm': preprocess_fix_misspell_lem,
}

vectorizer = vectorization()

res = {}
for prep_name, prep_func in preprocessors.items():
    X_train_prep = X_train.apply(lambda t: ' '.join(prep_func(t)))
    X_test_prep  = X_test.apply(lambda t: ' '.join(prep_func(t)))


    for vec_name, vec in vectorizer.items():
        if vec_name == 'hybrid':
                X_train_vec, X_test_vec = build_hybrid(X_train_prep, X_test_prep)
        else: 
            vec_copy = vec()

            X_train_vec = vec_copy.fit_transform(X_train_prep).astype(np.float32)
            X_test_vec = vec_copy.transform(X_test_prep).astype(np.float32)

        X_train_vec = X_train_vec.astype(np.float32)
        X_test_vec  = X_test_vec.astype(np.float32)

        model = LogisticRegression(max_iter=1000, class_weight='balanced')
        
        model.fit(X_train_vec, y_train)
        pred = model.predict(X_test_vec)
        acc = accuracy_score(y_test, pred)

        res[(prep_name, vec_name)] = acc
        print(f"{prep_name}, {vec_name} done")

fix, tfidf_no_stop done
fix, char done
fix, hybrid done
fix+misspell, tfidf_no_stop done
fix+misspell, char done
fix+misspell, hybrid done
fix+misspell+lemm, tfidf_no_stop done
fix+misspell+lemm, char done
fix+misspell+lemm, hybrid done


In [72]:
df_res = pd.DataFrame([(k[0], k[1], v) for k, v in res.items()], columns=['preprocessing', 'vectorization', 'accuracy'])

pivot = df_res.pivot(index='preprocessing', columns='vectorization', values='accuracy')
print(pivot.round(4))

vectorization        char  hybrid  tfidf_no_stop
preprocessing                                   
fix                0.8140  0.8174         0.7986
fix+misspell       0.8174  0.8294         0.8055
fix+misspell+lemm  0.8225  0.8362         0.7969


## Grid Search + top10 for hybrid

In [ ]:
preprocessors = {
    'fix': preprocess_fix,
    'fix+misspell': preprocess_fix_misspell,
    'fix+misspell+lemm': preprocess_fix_misspell_lem,
}

param_grid = {
    'C': [0.1, 0.3, 0.5, 1.0, 2.0, 5.0],
    'class_weight': ['balanced', None],
    'solver': ['liblinear', 'lbfgs'],
}

vectorizer = vectorization()

res = {}
all_sim = []
for prep_name, prep_func in preprocessors.items():
    X_train_prep = X_train.apply(lambda t: ' '.join(prep_func(t)))
    X_test_prep  = X_test.apply(lambda t: ' '.join(prep_func(t)))

    X_train_vec, X_test_vec = build_hybrid(X_train_prep, X_test_prep)

    grid = GridSearchCV(
        LogisticRegression(max_iter=1000),
        param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train_vec, y_train)
    
    best_model = grid.best_estimator_
    pred = best_model.predict(X_test_vec)
    acc  = accuracy_score(y_test, pred)

    res[prep_name] = {
        'best_params': grid.best_params_,
        'cv_score': round(grid.best_score_, 4),
        'test_acc': round(acc, 4),
    }
    df_sim = top_10(X_test_vec, X_test_prep)
    df_sim.insert(0, 'prep', prep_name)
    all_sim.append(df_sim)

    print(f"{prep_name}, {vec_name} done")

similarity_df = pd.concat(all_sim, ignore_index=True)

In [104]:
similarity_df.to_csv('similarity_best.csv', index=False)

In [103]:
df_res = pd.DataFrame([
    {
        'preprocessing': prep,
        'cv_score': info['cv_score'],
        'test_acc': info['test_acc'],
        'best_params': str(info['best_params']),
    }
    for prep, info in res.items()
])

print(df_res.to_string(index=False))

    preprocessing  cv_score  test_acc                                               best_params
              fix    0.8035    0.8208       {'C': 5.0, 'class_weight': None, 'solver': 'lbfgs'}
     fix+misspell    0.7988    0.8208       {'C': 5.0, 'class_weight': None, 'solver': 'lbfgs'}
fix+misspell+lemm    0.7971    0.8396 {'C': 2.0, 'class_weight': 'balanced', 'solver': 'lbfgs'}


# Bonus

In [26]:
def train_word2vec(text, vec_size=200, window=5, min_count=2, epochs=20):
    tokenized = [t.split() for t in text]
    model = Word2Vec(
        sentences=tokenized,
        vector_size=vec_size,
        min_count=min_count,
        window=window,
        workers=4,
        sg=1,
        epochs=epochs,
        seed=42
    )
    return model

def tweet_vec_mean(text, model, vector_size=200):
    words = text.split()
    vecs = [model.wv[w] for w in words if w in model.wv]
    if vecs:
        return np.mean(vecs, axis=0).astype(np.float32)
    return np.zeros(vector_size, dtype=np.float32)

In [22]:
X_train_prep = X_train.apply(lambda t: ' '.join(preprocess_fix_misspell_lem(t)))
X_test_prep  = X_test.apply(lambda t: ' '.join(preprocess_fix_misspell_lem(t)))

word2vec = train_word2vec(X_train_prep)

In [27]:
print(f"{len(word2vec.wv)}")

X_train_w2v = np.array([tweet_vec_mean(t, word2vec) for t in X_train_prep])
X_test_w2v = np.array([tweet_vec_mean(t, word2vec) for t in X_test_prep])

1945


In [36]:
model_w2v = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', C=7.0))
model_w2v.fit(X_train_w2v, y_train)
w2v_pred = model_w2v.predict(X_test_w2v)
w2c_acc = accuracy_score(w2v_pred, y_test)
w2c_acc

0.773037542662116

## word2vec + tfidf

In [37]:
X_train_tfidf, X_test_tfidf = build_hybrid(X_train_prep, X_test_prep)

X_train_w2v_sp = csr_matrix(X_train_w2v)
X_test_w2v_sp  = csr_matrix(X_test_w2v)

X_train_combined = hstack([X_train_tfidf, X_train_w2v_sp]).astype(np.float32)
X_test_combined  = hstack([X_test_tfidf,  X_test_w2v_sp]).astype(np.float32)

In [55]:
model = LogisticRegression(max_iter=1000, class_weight='balanced', C=9.0)
model.fit(X_train_combined, y_train)
pred = model.predict(X_test_combined)
acc = accuracy_score(pred, y_test)
acc

0.8191126279863481

че то все печально, и я пришел к выводу что нужно менять сам парсинг датасета

In [137]:
def parse_file_by_markers(filepath, class_name, markers):
    """сплит по маркерам а не по запятым"""
    with open(filepath, encoding='utf-8') as f:
        text = f.read()
    
    if markers:
        markers_re = '|'.join(re.escape(m) for m in markers)
        parts = re.split(rf'\s+(?:{markers_re})\s*,?\s*', text)
    else:
        parts = re.split(r',(?=\s*[A-Z])', text)
    
    result = [p.strip() for p in parts if p.strip() and len(p.strip()) > 10]
    return pd.DataFrame({'text': result, 'type': class_name})

In [138]:
markers = {
    'positive': ['happy', 'smile', 'crying with joy'],
    'negative': ['unhappy', 'sad', 'crying', 'silence', 'annoyance'],
}

data_pos = parse_file_by_markers('../datasets/p00_tweets/processedPositive.csv', 'positive', markers['positive'])
data_neg = parse_file_by_markers('../datasets/p00_tweets/processedNegative.csv', 'negative', markers['negative'])
data_neu = parse_file_by_markers('../datasets/p00_tweets/processedNeutral.csv', 'neutral',  [])

data = pd.concat([data_pos, data_neg, data_neu], ignore_index=True)
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['type'], test_size=0.2, random_state=42, stratify=data['type'])

In [97]:
X_train_prep = X_train.apply(lambda t: ' '.join(preprocess_fix_misspell_lem(t)))
X_test_prep  = X_test.apply(lambda t: ' '.join(preprocess_fix_misspell_lem(t)))

In [99]:
def build_hybrid_more(X_train_text, X_test_text):
    vec_w = TfidfVectorizer(
        analyzer='word', ngram_range=(1, 3),
        max_features=30000, sublinear_tf=True
    )
    X_train_w = vec_w.fit_transform(X_train_text).astype(np.float32)
    X_test_w  = vec_w.transform(X_test_text).astype(np.float32)

    vec_c = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(2, 6),
        max_features=50000, sublinear_tf=True
    )
    X_train_c = vec_c.fit_transform(X_train_text).astype(np.float32)
    X_test_c  = vec_c.transform(X_test_text).astype(np.float32)

    X_train = hstack([X_train_w, X_train_c])
    X_test  = hstack([X_test_w,  X_test_c])

    return X_train, X_test

In [132]:
X_train_vec, X_test_vec = build_hybrid_more(X_train, X_test)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    LogisticRegression(max_iter=3000, solver='lbfgs'),
    {'C': [1.0, 2.0, 3.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0],
     'class_weight': ['balanced', None]},
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train_vec, y_train)

best = grid.best_estimator_
print(grid.best_params_)
accuracy_score(y_test, best.predict(X_test_vec))

{'C': 6.0, 'class_weight': 'balanced'}


0.8645418326693227

In [145]:
def parse_file_with_markers(filepath, class_name):
    """маркеры остаются в тексте"""
    with open(filepath, encoding='utf-8') as f:
        text = f.read()
    
    parts = re.split(r',(?=\s*[A-Z])', text)
    result = []
    for p in parts:
        sub = re.split(r'(?<=[.!?])\s+(?=[A-Z])', p)
        result.extend(sub)
    
    result = [r.strip() for r in result if r.strip() and len(r.strip()) > 10]
    return pd.DataFrame({'text': result, 'type': class_name})


data_pos = parse_file('../datasets/p00_tweets/processedPositive.csv', 'positive')
data_neg = parse_file('../datasets/p00_tweets/processedNegative.csv', 'negative')
data_neu = parse_file('../datasets/p00_tweets/processedNeutral.csv', 'neutral')

data = pd.concat([data_pos, data_neg, data_neu], ignore_index=True)
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['type'], test_size=0.2, random_state=42, stratify=data['type'])

In [143]:
X_train_prep = X_train.tolist()
X_test_prep  = X_test.tolist()

X_train_vec, X_test_vec = build_hybrid_more(X_train_prep, X_test_prep)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    LogisticRegression(max_iter=3000, solver='lbfgs'),
    {'C': [1.0, 2.0, 3.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0],
     'class_weight': ['balanced', None]},
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train_vec, y_train)

best = grid.best_estimator_
print(grid.best_params_)
accuracy_score(y_test, best.predict(X_test_vec))

{'C': 8.0, 'class_weight': 'balanced'}


0.9180887372013652